# MedSegDiff — Asian Dataset Evaluation

Computes per-muscle Dice / Hausdorff / etc. for MedSegDiff segmentations on
the **MRI_data_asian** thigh dataset against the unilateral ground truth.

Segmentation files are expected at:
`../asian_segs/{subject}/Thigh/Thigh_seg.npz`

Results are saved to `results_asian/`.

## Muscle mapping

MedSegDiff was trained on myosegmenTUM with bilateral labels (R and L separate).
The Asian GT is **unilateral** — each label covers only one anatomical side.
Both R and L predictions are therefore evaluated against the same GT label;
the side with higher Dice reveals which side is annotated.

| NPZ key | Asian GT label | Muscle |
|---|---|---|
| `R_gracilis` | 6 | gracilis |
| `L_gracilis` | 6 | gracilis |
| `R_sartorius` | 5 | sartorius |
| `L_sartorius` | 5 | sartorius |

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

DATA_ROOT  = os.path.join('..', '..', 'MRI_data_asian', 'MRI_data')
SEG_DIR    = os.path.join('..', 'asian_segs')
RESULT_DIR = 'results_asian'

os.makedirs(RESULT_DIR, exist_ok=True)

# (output_csv_name, asian_gt_label, npz_key)
# Both R and L are evaluated against the same unilateral GT label.
MUSCLES = [
    ('R_gracilis',  6, 'R_gracilis'),
    ('L_gracilis',  6, 'L_gracilis'),
    ('R_sartorius', 5, 'R_sartorius'),
    ('L_sartorius', 5, 'L_sartorius'),
]

npz_files = sorted(glob.glob(os.path.join(SEG_DIR, '*', 'Thigh', 'Thigh_seg.npz')))
print(f'DATA_ROOT : {os.path.abspath(DATA_ROOT)}  exists={os.path.isdir(DATA_ROOT)}')
print(f'SEG_DIR   : {os.path.abspath(SEG_DIR)}  exists={os.path.isdir(SEG_DIR)}')
print(f'NPZ files : {len(npz_files)}')
for f in npz_files[:5]:
    print(f'  {f}')

In [ ]:
def evaluate_muscle(muscle_name, gt_label, npz_key, npz_files, result_dir):
    """
    Evaluate one muscle across all subjects.
    npz_files: list of paths like .../asian_segs/{subject}/Thigh/Thigh_seg.npz
    """
    results = []
    for npz_path in npz_files:
        parts   = npz_path.replace('\\', '/').split('/')
        subject = parts[-3]

        gt_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'mask_muscles.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        npz_data = np.load(npz_path)
        if npz_key not in npz_data.files:
            print(f'  [{subject}] key "{npz_key}" not in NPZ, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        pred_arr  = npz_data[npz_key].astype(float)
        pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  [{subject}] empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'subject':                              subject,
            'seg_file':                             os.path.basename(npz_path),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_medsegdiff_asian.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df


print('evaluate_muscle ready.')

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────
dfs = {}
for muscle_name, gt_label, npz_key in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, key="{npz_key}") ──')
    dfs[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, npz_key, npz_files, RESULT_DIR,
    )
print('\nDone.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
from IPython.display import display

summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty:
        continue
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[dice_col].mean(),
        'dice_std':       df[dice_col].std(),
        'hausdorff_mean': df[hd_col].mean(),
        'hausdorff_std':  df[hd_col].std(),
    })

summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, 'summary_medsegdiff_asian.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))

In [ ]:
# ── R vs L Dice comparison ────────────────────────────────────────────────────
# Since the Asian GT is unilateral, the higher-Dice side reveals which side
# was annotated.  This cell shows R and L side-by-side per muscle type.
for base in ('gracilis', 'sartorius'):
    r_name = f'R_{base}'
    l_name = f'L_{base}'
    if r_name in dfs and l_name in dfs and not dfs[r_name].empty:
        r_dice = dfs[r_name][f'{r_name}_dice'].mean()
        l_dice = dfs[l_name][f'{l_name}_dice'].mean()
        labeled = 'R' if r_dice >= l_dice else 'L'
        print(f'{base}: R_dice={r_dice:.4f}  L_dice={l_dice:.4f}  '
              f'→ labeled side appears to be {labeled}')